# Modulo 3 · Lezione 5 — Progetto: automazione con Bash

**Corso di Intelligenza Artificiale — Livello base**

---

## Obiettivo del progetto

In questo progetto costruiamo **un sistema di automazione reale** applicando tutto quello visto nel Modulo 3. Il sistema gestisce esperimenti di Machine Learning: organizza i dati, lancia gli addestramenti, analizza i log e fa backup automatici.

```
SISTEMA DI AUTOMAZIONE ML

setup.sh           → prepara l'ambiente del progetto
lanzar.sh          → lancia un esperimento (config + simulazione)
analizar.sh        → analizza i log di addestramento
backup.sh          → backup con rotazione automatica
estado.sh          → report completo del progetto
```

Ogni script usa gli strumenti delle quattro lezioni precedenti:

| Script | Competenze usate |
|--------|-------------------|
| `setup.sh` | `mkdir -p`, `chmod`, permessi, condizionali |
| `lanzar.sh` | Argomenti, variabili, cicli, heredoc |
| `analizar.sh` | `grep`, `find`, pipe, redirezioni |
| `backup.sh` | `cp`/`mv`, `find -mtime`, funzioni, rotazione |
| `estado.sh` | Tutto combinato, funzioni, report finale |

---
## Fase 1 — Struttura del progetto

In [ ]:
import os, shutil
from pathlib import Path

PROYECTO = Path.home() / "ml_auto"
if PROYECTO.exists(): shutil.rmtree(PROYECTO)

# Struttura base
for d in [
    "scripts", "datos/raw", "datos/procesados",
    "experimentos", "logs", "backups", "config"
]:
    (PROYECTO / d).mkdir(parents=True)

os.chdir(PROYECTO)
print(f"Progetto in: {PROYECTO}")
!ls -R

---
## Fase 2 — Script `setup.sh`

In [ ]:
Path("scripts/setup.sh").write_text("""\
#!/usr/bin/env bash
# setup.sh — prepara l'ambiente del progetto ML
set -euo pipefail

PROYECTO_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.."; pwd)"

log() { echo "[SETUP] $*"; }

log "Inizializzazione progetto in: $PROYECTO_DIR"

# Creare le directory necessarie
DIRS=("datos/raw" "datos/procesados" "experimentos" "logs" "backups" "config")
for dir in "${DIRS[@]}"; do
    ruta="$PROYECTO_DIR/$dir"
    if [ ! -d "$ruta" ]; then
        mkdir -p "$ruta"
        log "  Creata: $dir"
    else
        log "  OK:     $dir"
    fi
done

# Rendere eseguibili tutti gli script
chmod +x "$PROYECTO_DIR/scripts"/*.sh
log "Script contrassegnati come eseguibili"

# Creare la config di default se non esiste
CONFIG="$PROYECTO_DIR/config/defaults.yaml"
if [ ! -f "$CONFIG" ]; then
    cat > "$CONFIG" << 'EOF'
# Configurazione di default
epochs: 50
batch_size: 32
learning_rate: 0.001
optimizer: adam
max_backups: 5
log_level: INFO
EOF
    log "Config di default creata: $CONFIG"
fi

# Verificare gli strumenti
log "Verifica strumenti..."
HERRAMIENTAS=(python3 git grep find)
OK=0; FALLOS=0
for tool in "${HERRAMIENTAS[@]}"; do
    if command -v "$tool" &>/dev/null; then
        log "  ✅ $tool"
        ((OK++))
    else
        log "  ❌ $tool — non trovato"
        ((FALLOS++))
    fi
done

echo ""
log "Setup completato: $OK OK, $FALLOS falliti"
[ $FALLOS -eq 0 ] && exit 0 || exit 1
""", encoding="utf-8")

!chmod +x scripts/setup.sh
print("Esecuzione di setup.sh:")
!bash scripts/setup.sh

---
## Fase 3 — Script `lanzar.sh`

In [ ]:
Path("scripts/lanzar.sh").write_text("""\
#!/usr/bin/env bash
# lanzar.sh — lancia un esperimento di addestramento
# Uso: lanzar.sh <nome> <lr> [epochs] [batch_size]

PROYECTO_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.."; pwd)"

NOMBRE=${1:?"Argomento 1 richiesto: nome dell'esperimento"}
LR=${2:?"Argomento 2 richiesto: learning rate"}
EPOCHS=${3:-50}
BATCH=${4:-32}

FECHA=$(date +%Y%m%d_%H%M%S)
EXP_DIR="$PROYECTO_DIR/experimentos/${NOMBRE}_${FECHA}"
LOG="$PROYECTO_DIR/logs/${NOMBRE}_${FECHA}.log"

log() { echo "[$(date +%H:%M:%S)] $*" | tee -a "$LOG"; }

# Creare la directory dell'esperimento
mkdir -p "$EXP_DIR"

log "=== ESPERIMENTO: $NOMBRE ==="
log "Data:           $FECHA"
log "Learning rate:  $LR"
log "Epochs:         $EPOCHS"
log "Batch size:     $BATCH"

# Salvare la configurazione
cat > "$EXP_DIR/config.yaml" << EOF
experimento: $NOMBRE
fecha: $FECHA
learning_rate: $LR
epochs: $EPOCHS
batch_size: $BATCH
EOF
log "Config salvata in: $EXP_DIR/config.yaml"

# Simulare l'addestramento
log "Avvio addestramento..."
import_err=0
for ((epoch=1; epoch<=EPOCHS; epoch+=10)); do
    LOSS=$(echo "$LR $epoch" | awk '{printf "%.4f", 2.0 * exp(-$2 * 0.05 * $1 * 100)}')
    ACC=$(echo "$LOSS" | awk '{printf "%.4f", 1 - $1/2.5}')
    log "[INFO] Epoch $(printf '%02d' $epoch)/$EPOCHS — loss=$LOSS acc=$ACC"
    if [ $((epoch % 30)) -eq 0 ]; then
        log "[WARNING] Plateau rilevato all'epoch $epoch"
    fi
done

log "Addestramento completato"
log "Log salvato in: $LOG"

echo ""
echo "✅ Esperimento $NOMBRE completato"
echo "   Directory: $EXP_DIR"
echo "   Log:       $LOG"
""", encoding="utf-8")

!chmod +x scripts/lanzar.sh

print("Lancio degli esperimenti:")
!bash scripts/lanzar.sh modelo_v1 0.001 50 32
print("\n---")
!bash scripts/lanzar.sh modelo_v2 0.01 50 64
print("\n---")
!bash scripts/lanzar.sh modelo_v3 0.0001 50 16

---
## Fase 4 — Script `analizar.sh`

In [ ]:
Path("scripts/analizar.sh").write_text("""\
#!/usr/bin/env bash
# analizar.sh — analizza i log di addestramento

PROYECTO_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.."; pwd)"
LOGS_DIR="$PROYECTO_DIR/logs"
FECHA=$(date +%Y%m%d_%H%M%S)
INFORME="$PROYECTO_DIR/logs/analisis_${FECHA}.txt"

if [ ! -d "$LOGS_DIR" ]; then
    echo "Errore: directory dei log non trovata" >&2
    exit 1
fi

LOGS=$(find "$LOGS_DIR" -name "*.log" -not -name "analisis_*" -type f | sort)
TOTAL_LOGS=$(echo "$LOGS" | grep -c '.' 2>/dev/null || echo 0)

if [ "$TOTAL_LOGS" -eq 0 ]; then
    echo "Nessun log di addestramento trovato"
    exit 0
fi

{
echo "================================================"
echo "  ANALISI ESPERIMENTI — $(date)"
echo "================================================"
echo "Log trovati: $TOTAL_LOGS"
echo ""

printf '%-35s %7s %7s %7s\n' "ESPERIMENTO" "EPOCHS" "WARNS" "FINAL_ACC"
printf '%s\n' "$(printf '─%.0s' {1..60})"

MEJOR_EXP=""
MEJOR_ACC=0

while IFS= read -r log; do
    [ -z "$log" ] && continue
    nombre=$(basename "$log" .log)
    epochs=$(grep -c '\[INFO\]' "$log" 2>/dev/null || echo 0)
    warnings=$(grep -c '\[WARNING\]' "$log" 2>/dev/null || echo 0)
    ultima_acc=$(grep -oE 'acc=[0-9.]+' "$log" | tail -1 | cut -d= -f2)
    ultima_acc=${ultima_acc:-"0.0000"}

    printf '%-35s %7s %7s %7s\n' "$nombre" "$epochs" "$warnings" "$ultima_acc"

    # Tracciamento del migliore
    if awk "BEGIN {exit !($ultima_acc > $MEJOR_ACC)}" 2>/dev/null; then
        MEJOR_ACC=$ultima_acc
        MEJOR_EXP=$nombre
    fi
done <<< "$LOGS"

echo ""
echo "MIGLIOR ESPERIMENTO:"
echo "  $MEJOR_EXP (acc=$MEJOR_ACC)"

echo ""
echo "AVVERTIMENTI TOTALI:"
cat $LOGS_DIR/*.log 2>/dev/null | grep -c '\[WARNING\]' | xargs echo " "

echo ""
echo "ERRORI TROVATI:"
ERRORES=$(cat $LOGS_DIR/*.log 2>/dev/null | grep '\[ERROR\]' || echo "  (nessuno)")
echo "$ERRORES"

} | tee "$INFORME"

echo ""
echo "Report salvato: $INFORME"
""", encoding="utf-8")

!chmod +x scripts/analizar.sh
print("Esecuzione analisi:")
!bash scripts/analizar.sh

---
## Fase 5 — Script `backup.sh`

In [ ]:
Path("scripts/backup.sh").write_text("""\
#!/usr/bin/env bash
# backup.sh — backup con rotazione automatica

PROYECTO_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.."; pwd)"
BACKUPS_DIR="$PROYECTO_DIR/backups"
MAX_BACKUPS=${1:-5}
FECHA=$(date +%Y%m%d_%H%M%S)
NOMBRE="backup_${FECHA}.tar.gz"
LOG_BACKUP="$BACKUPS_DIR/backup.log"

log() { echo "[$(date +%H:%M:%S)] $*" | tee -a "$LOG_BACKUP"; }

log "=== BACKUP $FECHA ==="
log "Progetto:     $PROYECTO_DIR"
log "Destinazione: $BACKUPS_DIR/$NOMBRE"

# Creare il backup escludendo i backup precedenti e le cache
tar -czf "$BACKUPS_DIR/$NOMBRE" \
    --exclude="$PROYECTO_DIR/backups" \
    --exclude="*/__pycache__" \
    --exclude="*.pyc" \
    -C "$(dirname "$PROYECTO_DIR")" \
    "$(basename "$PROYECTO_DIR")" 2>/dev/null

if [ -s "$BACKUPS_DIR/$NOMBRE" ]; then
    TAMANO=$(du -sh "$BACKUPS_DIR/$NOMBRE" | cut -f1)
    log "Backup creato: $NOMBRE ($TAMANO)"
else
    log "ERRORE: creazione backup fallita"
    exit 1
fi

# Rotazione
TOTAL=$(find "$BACKUPS_DIR" -maxdepth 1 -name 'backup_*.tar.gz' | wc -l)
log "Backup esistenti: $TOTAL (massimo: $MAX_BACKUPS)"
ELIMINADOS=0

while [ "$TOTAL" -gt "$MAX_BACKUPS" ]; do
    ANTIGUO=$(find "$BACKUPS_DIR" -maxdepth 1 -name 'backup_*.tar.gz' | sort | head -1)
    log "Eliminazione più vecchio: $(basename "$ANTIGUO")"
    rm "$ANTIGUO"
    ((TOTAL--))
    ((ELIMINADOS++))
done

log "Eliminati: $ELIMINADOS | Attuali: $(find "$BACKUPS_DIR" -maxdepth 1 -name 'backup_*.tar.gz' | wc -l)"
log "=== FINE BACKUP OK ==="

echo ""
echo "✅ Backup: $NOMBRE ($TAMANO)"
""", encoding="utf-8")

!chmod +x scripts/backup.sh
print("Esecuzione backup:")
!bash scripts/backup.sh 3

print("\nContenuto di backups/:")
!ls -lh backups/

---
## Fase 6 — Script `estado.sh` (l'integratore)

In [ ]:
Path("scripts/estado.sh").write_text("""\
#!/usr/bin/env bash
# estado.sh — report completo dello stato del progetto

PROYECTO_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.."; pwd)"
NOMBRE_PROYECTO=$(basename "$PROYECTO_DIR")

separador() { printf '%s\n' "$(printf '═%.0s' {1..55})"; }
seccion()    { echo ""; separador; echo "  $1"; separador; }

separador
echo "  STATO DEL PROGETTO: $NOMBRE_PROYECTO"
echo "  Data: $(date '+%Y-%m-%d %H:%M:%S')"
separador

# ── Struttura ──────────────────────────────────────────────
seccion "STRUTTURA"
TOTAL_ARCH=$(find "$PROYECTO_DIR" -type f | wc -l)
TOTAL_DIR=$(find "$PROYECTO_DIR" -type d | wc -l)
TAMANO=$(du -sh "$PROYECTO_DIR" 2>/dev/null | cut -f1)
echo "  File:         $TOTAL_ARCH"
echo "  Directory:    $TOTAL_DIR"
echo "  Dimensione totale: $TAMANO"

# ── Script ─────────────────────────────────────────────────
seccion "SCRIPT (.sh)"
for script in "$PROYECTO_DIR/scripts"/*.sh; do
    [ -f "$script" ] || continue
    nombre=$(basename "$script")
    lineas=$(wc -l < "$script")
    ejecutable=$([ -x "$script" ] && echo "✅" || echo "❌")
    printf '  %s %-25s %3d righe\n' "$ejecutable" "$nombre" "$lineas"
done

# ── Esperimenti ────────────────────────────────────────────
seccion "ESPERIMENTI"
EXP_COUNT=$(find "$PROYECTO_DIR/experimentos" -maxdepth 1 -mindepth 1 -type d | wc -l)
echo "  Totale: $EXP_COUNT"
if [ "$EXP_COUNT" -gt 0 ]; then
    echo "  Ultimi 3:"
    find "$PROYECTO_DIR/experimentos" -maxdepth 1 -mindepth 1 -type d | \
        sort -r | head -3 | while read -r exp; do
        echo "    $(basename "$exp")"
    done
fi

# ── Log ────────────────────────────────────────────────────
seccion "LOG"
LOG_COUNT=$(find "$PROYECTO_DIR/logs" -name "*.log" -not -name "analisis_*" | wc -l)
echo "  Log di addestramento: $LOG_COUNT"
if [ "$LOG_COUNT" -gt 0 ]; then
    TOTAL_ERRORES=$(cat "$PROYECTO_DIR"/logs/*.log 2>/dev/null | grep -c '\[ERROR\]' || echo 0)
    TOTAL_WARNINGS=$(cat "$PROYECTO_DIR"/logs/*.log 2>/dev/null | grep -c '\[WARNING\]' || echo 0)
    echo "  Errori totali:        $TOTAL_ERRORES"
    echo "  Warning totali:       $TOTAL_WARNINGS"
    if [ "$TOTAL_ERRORES" -gt 0 ]; then
        echo "  ⚠️  Ci sono errori — esegui analizar.sh per i dettagli"
    fi
fi

# ── Backup ─────────────────────────────────────────────────
seccion "BACKUP"
BACKUP_COUNT=$(find "$PROYECTO_DIR/backups" -name 'backup_*.tar.gz' | wc -l)
echo "  Backup disponibili: $BACKUP_COUNT"
if [ "$BACKUP_COUNT" -gt 0 ]; then
    echo "  Più recente:"
    find "$PROYECTO_DIR/backups" -name 'backup_*.tar.gz' | sort -r | head -1 | \
        xargs -I{} sh -c 'echo "    $(basename {}) — $(du -sh {} | cut -f1)"'
fi

# ── Config ─────────────────────────────────────────────────
seccion "CONFIGURAZIONE"
CONFIG="$PROYECTO_DIR/config/defaults.yaml"
if [ -f "$CONFIG" ]; then
    echo "  defaults.yaml:"
    grep -v '^#' "$CONFIG" | grep -v '^$' | sed 's/^/    /'
else
    echo "  ⚠️  defaults.yaml non esiste — esegui setup.sh"
fi

separador
echo "  Comandi disponibili:"
echo "    bash scripts/setup.sh              # inizializzare"
echo "    bash scripts/lanzar.sh <n> <lr>    # nuovo esperimento"
echo "    bash scripts/analizar.sh           # analizzare i log"
echo "    bash scripts/backup.sh [max]       # fare backup"
separador
""", encoding="utf-8")

!chmod +x scripts/estado.sh
print("Stato del progetto:")
!bash scripts/estado.sh

---
## Pratica — Estendere il sistema

---

### Esercizio 1 — Script `comparar.sh`

**Consegna:** Scrivi `scripts/comparar.sh` che confronta due esperimenti e determina quale è stato migliore:

```bash
bash scripts/comparar.sh esperimento_A esperimento_B
```

Lo script deve:
1. Ricevere i nomi di due esperimenti come argomenti.
2. Cercare i loro log in `logs/`.
3. Estrarre: loss iniziale, loss finale, accuracy finale, numero di warning.
4. Mostrare una tabella comparativa.
5. Dichiarare un vincitore in base all'accuracy finale.

In [ ]:
# ✏️ IL TUO CODICE QUI
from pathlib import Path
import os
os.chdir(Path.home() / "ml_auto")


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```bash
#!/usr/bin/env bash
# comparar.sh — confronta due esperimenti

PROYECTO_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")/.."; pwd)"
EXP_A=${1:?"Argomento 1: nome dell'esperimento A"}
EXP_B=${2:?"Argomento 2: nome dell'esperimento B"}

# Cercare il log di un esperimento
buscar_log() {
    local nombre="$1"
    find "$PROYECTO_DIR/logs" -name "${nombre}*.log" -not -name "analisis_*" | head -1
}

# Estrarre le metriche da un log
extraer_metricas() {
    local log="$1"
    local loss_inicial warnings acc_final loss_final

    loss_inicial=$(grep -oE 'loss=[0-9.]+' "$log" | head -1 | cut -d= -f2)
    loss_final=$(grep -oE 'loss=[0-9.]+' "$log" | tail -1 | cut -d= -f2)
    acc_final=$(grep -oE 'acc=[0-9.]+' "$log" | tail -1 | cut -d= -f2)
    warnings=$(grep -c '\[WARNING\]' "$log" 2>/dev/null || echo 0)

    echo "$loss_inicial $loss_final $acc_final $warnings"
}

LOG_A=$(buscar_log "$EXP_A")
LOG_B=$(buscar_log "$EXP_B")

[ -z "$LOG_A" ] && { echo "Errore: nessun log trovato per '$EXP_A'" >&2; exit 1; }
[ -z "$LOG_B" ] && { echo "Errore: nessun log trovato per '$EXP_B'" >&2; exit 1; }

read -r L0A LFA ACCA WA <<< "$(extraer_metricas "$LOG_A")"
read -r L0B LFB ACCB WB <<< "$(extraer_metricas "$LOG_B")"

printf '\n%-20s %10s %10s %12s %8s\n' "ESPERIMENTO" "LOSS_INI" "LOSS_FIN" "ACC_FINAL" "WARNS"
printf '%s\n' "$(printf '─%.0s' {1..62})"
printf '%-20s %10s %10s %12s %8s\n' "$EXP_A" "$L0A" "$LFA" "$ACCA" "$WA"
printf '%-20s %10s %10s %12s %8s\n' "$EXP_B" "$L0B" "$LFB" "$ACCB" "$WB"
echo ""

if awk "BEGIN {exit !(${ACCA:-0} > ${ACCB:-0})}" 2>/dev/null; then
    echo "🏆 Vincitore: $EXP_A (acc=$ACCA vs $ACCB)"
elif awk "BEGIN {exit !(${ACCB:-0} > ${ACCA:-0})}" 2>/dev/null; then
    echo "🏆 Vincitore: $EXP_B (acc=$ACCB vs $ACCA)"
else
    echo "🤝 Pareggio tecnico"
fi
```
</details>

---
### Esercizio 2 — Script `limpiar.sh`

**Consegna:** Scrivi `scripts/limpiar.sh` che elimina file temporanei ed esperimenti vecchi:

1. Elimina i file `.pyc`, `__pycache__/` e `.tmp` dal progetto.
2. Elenca gli esperimenti con più di N giorni di anzianità (parametro opzionale, di default 7).
3. Chiede conferma prima di eliminare gli esperimenti vecchi.
4. Riporta lo spazio recuperato.
5. Registra la pulizia in `logs/limpieza.log`.

In [ ]:
# ✏️ IL TUO CODICE QUI


---
### Esercizio 3 — `cron` e automazione periodica

**Consegna (concettuale + pratica):**

`cron` è lo scheduler di attività di Linux. Ti permette di eseguire script automaticamente a un orario o intervallo determinato.

```bash
# Vedere le attività cron dell'utente
crontab -l

# Modificare
crontab -e

# Formato: minuto ora giorno_mese mese giorno_settimana comando
#  *   *   *   *   *   comando
#  │   │   │   │   └── giorno settimana (0=dom, 7=dom)
#  │   │   │   └────── mese (1-12)
#  │   │   └────────── giorno del mese (1-31)
#  │   └────────────── ora (0-23)
#  └────────────────── minuto (0-59)
```

**Esempi:**
```bash
0 2 * * *  /home/user/ml_auto/scripts/backup.sh        # backup ogni giorno alle 2:00
*/15 * * * * /home/user/ml_auto/scripts/analizar.sh    # analisi ogni 15 minuti
0 8 * * 1  /home/user/ml_auto/scripts/estado.sh        # report ogni lunedì alle 8:00
```

**Compito:**
1. Scrivi uno script `instalar_cron.sh` che aggiunga una voce cron per fare il backup giornaliero alle 3:00.
2. Verifica con `crontab -l` che sia stata aggiunta correttamente.
3. Scrivi il comando per eliminare quella voce specifica senza cancellare le altre.

In [ ]:
# ✏️ IL TUO CODICE QUI — instalar_cron.sh


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```bash
#!/usr/bin/env bash
# instalar_cron.sh — aggiunge un'attività cron di backup

SCRIPT="$HOME/ml_auto/scripts/backup.sh"
LOG="$HOME/ml_auto/logs/cron_backup.log"
ENTRADA="0 3 * * * $SCRIPT >> $LOG 2>&1"

# Verificare che lo script esista
if [ ! -f "$SCRIPT" ]; then
    echo "Errore: $SCRIPT non esiste" >&2
    exit 1
fi

# Aggiungere solo se non esiste già
if crontab -l 2>/dev/null | grep -qF "$SCRIPT"; then
    echo "La voce esiste già nel crontab"
else
    (crontab -l 2>/dev/null; echo "$ENTRADA") | crontab -
    echo "✅ Voce aggiunta:"
    echo "   $ENTRADA"
fi

echo ""
echo "Crontab attuale:"
crontab -l
```

**Per eliminare quella voce specifica:**
```bash
crontab -l | grep -vF "backup.sh" | crontab -
```

**Nota importante:** usa sempre percorsi assoluti nel crontab — quando cron esegue lo script, il `$PATH` è diverso da quello della tua shell interattiva e i programmi con percorsi relativi non vengono trovati.
</details>

---
### Esercizio 4 — Integrazione Python + Bash

**Consegna:** Scrivi uno script Python `orquestar.py` che usi `subprocess` per:

1. Chiamare `lanzar.sh` con tre learning rate distinti in sequenza.
2. Aspettare che ognuno finisca prima di lanciare il successivo.
3. Chiamare `analizar.sh` alla fine.
4. Leggere il report generato e mostrare a schermo solo le righe del riepilogo.
5. Restituire il nome del miglior esperimento.

In [ ]:
# ✏️ IL TUO CODICE QUI
import subprocess
from pathlib import Path


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import subprocess, re, os
from pathlib import Path

ML_AUTO = Path.home() / "ml_auto"
os.chdir(ML_AUTO)

def run(cmd, **kwargs):
    print(f"\n$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kwargs)
    if r.stdout: print(r.stdout.rstrip())
    if r.stderr: print(r.stderr.rstrip())
    return r

# 1 e 2. Lanciare gli esperimenti
experimentos = [
    ("grid_lr_001",   "0.001"),
    ("grid_lr_005",   "0.005"),
    ("grid_lr_01",    "0.01"),
]

for nombre, lr in experimentos:
    print(f"\n{'─'*40}")
    print(f"Lancio: {nombre} (lr={lr})")
    r = run(f"bash scripts/lanzar.sh {nombre} {lr} 30")
    if r.returncode != 0:
        print(f"⚠️  Esperimento {nombre} fallito")

# 3. Analizzare
print(f"\n{'─'*40}")
print("Analisi dei risultati...")
run("bash scripts/analizar.sh")

# 4. Leggere il report
informes = sorted(ML_AUTO.glob("logs/analisis_*.txt"), reverse=True)
if informes:
    contenido = informes[0].read_text(encoding="utf-8")
    print("\nRIEPILOGO DEL REPORT:")
    for linea in contenido.splitlines():
        if "MEJOR" in linea or "Total" in linea or linea.startswith("="):
            print(f"  {linea}")

    # 5. Estrarre il miglior esperimento
    m = re.search(r'MEJOR EXPERIMENTO:\s*\n\s*(\S+)', contenido)
    if m:
        print(f"\n🏆 Miglior esperimento: {m.group(1)}")
        return_val = m.group(1)
    else:
        return_val = None
        print("Non è stato possibile determinare il migliore")
```
</details>

---
### Esercizio 5 (sfida) — Makefile

**Consegna:** Un `Makefile` è il modo classico di documentare e concatenare comandi nei progetti software. È un'alternativa più portabile agli script di setup.

Crea un `Makefile` nella radice di `ml_auto/` con i seguenti target:

```makefile
make setup          # esegue setup.sh
make experimento N=nome LR=0.001   # lancia un esperimento
make analizar       # esegue analizar.sh
make backup         # esegue backup.sh
make estado         # esegue estado.sh
make limpiar        # elimina .pyc e __pycache__
make help           # mostra i comandi disponibili
```

Dopo, esegui `make help` e `make estado`.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```makefile
# Makefile — Sistema di automazione ML
.PHONY: setup experimento analizar backup estado limpiar help

SCRIPTS = scripts
N      ?= experimento_$(shell date +%H%M%S)
LR     ?= 0.001
EPOCHS ?= 50
MAX_BK ?= 5

help:
	@echo ""
	@echo "Sistema di Automazione ML"
	@echo "============================="
	@echo "  make setup                   Inizializzare il progetto"
	@echo "  make experimento N=x LR=0.01 Lanciare un esperimento"
	@echo "  make analizar                Analizzare i log"
	@echo "  make backup                  Fare un backup"
	@echo "  make estado                  Vedere lo stato del progetto"
	@echo "  make limpiar                 Eliminare i file temporanei"
	@echo ""

setup:
	bash $(SCRIPTS)/setup.sh

experimento:
	bash $(SCRIPTS)/lanzar.sh $(N) $(LR) $(EPOCHS)

analizar:
	bash $(SCRIPTS)/analizar.sh

backup:
	bash $(SCRIPTS)/backup.sh $(MAX_BK)

estado:
	bash $(SCRIPTS)/estado.sh

limpiar:
	find . -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null; true
	find . -name '*.pyc' -delete 2>/dev/null; true
	find . -name '*.tmp' -delete 2>/dev/null; true
	@echo "Pulizia completata"
```

```python
# Creare e testare da Python
from pathlib import Path
import os

MAKEFILE = Path.home() / "ml_auto" / "Makefile"
# (scrivi il contenuto sopra)

os.chdir(Path.home() / "ml_auto")
!make help
!make estado
```

**`?=`** definisce una variabile con valore di default solo se non è stata già definita. **`.PHONY`** indica che quei target non generano file — senza di esso, se esistesse un file chiamato `setup`, `make setup` non farebbe nulla.
</details>

---
## Riflessione finale — Cosa abbiamo costruito?

Il sistema `ml_auto/` è un'infrastruttura reale. Con cinque script hai automatizzato:

| Attività | Senza automazione | Con automazione |
|-------|------------------|-----------------|
| Preparare l'ambiente | Creare ogni cartella a mano | `bash setup.sh` |
| Lanciare esperimenti | Digitare i parametri ogni volta | `bash lanzar.sh nome 0.001` |
| Analizzare i risultati | Leggere i log uno per uno | `bash analizar.sh` |
| Fare backup | Comprimere e spostare a mano | `bash backup.sh` (o cron) |
| Vedere lo stato | Esplorare le directory | `bash estado.sh` |

> Automatizzare è un investimento: costa tempo scrivere lo script, ma lo recuperi ogni volta che lo esegui.

---
## Modulo 3 completato — Riepilogo di ciò che hai imparato

```
LEZIONE 1  Filesystem e navigazione
         ├── Struttura del filesystem Linux (/bin, /etc, /home…)
         ├── Percorsi assoluti e relativi (., .., ~, -)
         ├── pwd, ls -la, cd, mkdir -p, rmdir
         └── tree ed esplorazione dei progetti

LEZIONE 2  Gestione di file e permessi
         ├── cp, mv, rm (con e senza -r)
         ├── cat, head, tail, less, wc
         ├── touch, nano
         └── Permessi rwx, chmod 755/644/600, chown

LEZIONE 3  Ricerca ed elaborazione del testo
         ├── grep: -i, -n, -r, -v, -c, -E, -C
         ├── find: -name, -type, -mtime, -size, -exec, -delete
         ├── Pipe |: concatenare comandi
         ├── Redirezioni: >, >>, 2>, 2>&1, /dev/null
         └── wc, sort, uniq, cut

LEZIONE 4  Bash scripting
         ├── Shebang, set -euo pipefail, chmod +x
         ├── Variabili, sostituzione $(), aritmetica $(())
         ├── Argomenti $1 $2 $@ $# shift
         ├── if/elif/else, operatori -f -d -z -eq -gt
         ├── for (liste, glob, seq) e while
         └── Funzioni con local e return

LEZIONE 5  Progetto: sistema di automazione ML
         ├── setup.sh, lanzar.sh, analizar.sh
         ├── backup.sh con rotazione automatica
         ├── estado.sh integratore
         ├── cron per attività periodiche
         └── Makefile come interfaccia unificata
```

**Prossimo modulo:** Python per i dati — NumPy, Pandas e visualizzazione.